# Week 5: 模型比较

## 学习目标

1. 理解交叉验证的原理和作用
2. 掌握模型选择和超参数调优方法
3. 学会比较不同模型的性能
4. 理解偏差-方差权衡

## 1. 交叉验证

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("模型比较工具已加载")

### 1.1 为什么需要交叉验证？

In [ ]:
# 生成模拟数据
n_samples = 200
n_features = 10

X = np.random.randn(n_samples, n_features)
# 真实模型：只用前 3 个特征
true_coef = np.array([5, 3, -2, 0, 0, 0, 0, 0, 0, 0])
y = X @ true_coef + np.random.normal(0, 2, n_samples)

print(f"样本数: {n_samples}")
print(f"特征数: {n_features}")

In [ ]:
# 单次分割的问题：结果不稳定
test_scores = []

for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    test_scores.append(score)

print("不同随机种子下的测试集 R²")
print("=" * 40)
print(f"均值: {np.mean(test_scores):.4f}")
print(f"标准差: {np.std(test_scores):.4f}")
print(f"范围: [{min(test_scores):.4f}, {max(test_scores):.4f}]")

### 1.2 K-Fold 交叉验证

In [ ]:
# 5-Fold 交叉验证
model = LinearRegression()
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print("5-Fold 交叉验证结果")
print("=" * 40)
print(f"各折 R²: {cv_scores}")
print(f"均值: {np.mean(cv_scores):.4f}")
print(f"标准差: {np.std(cv_scores):.4f}")

In [ ]:
# 可视化交叉验证过程
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 单次分割 vs 交叉验证
axes[0].bar(range(10), test_scores, alpha=0.7, label='单次分割')
axes[0].axhline(np.mean(test_scores), color='red', linestyle='--', label=f'均值={np.mean(test_scores):.3f}')
axes[0].set_xlabel('随机种子')
axes[0].set_ylabel('R²')
axes[0].set_title('单次分割的变异性')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(5), cv_scores, alpha=0.7, color='green')
axes[1].axhline(np.mean(cv_scores), color='red', linestyle='--', label=f'均值={np.mean(cv_scores):.3f}')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('R²')
axes[1].set_title('5-Fold 交叉验证')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 模型比较

### 2.1 多种模型对比

In [ ]:
# 准备数据
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 定义模型
models = {
    '线性回归': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    '决策树': DecisionTreeRegressor(max_depth=5, random_state=42),
    '随机森林': RandomForestRegressor(n_estimators=100, random_state=42)
}

In [ ]:
# 训练和评估
results = []

for name, model in models.items():
    # 训练
    model.fit(X_train_scaled, y_train)
    
    # 预测
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # 评估
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    # 交叉验证
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    
    results.append({
        '模型': name,
        '训练 R²': train_r2,
        '测试 R²': test_r2,
        '测试 MSE': test_mse,
        'CV R² 均值': np.mean(cv_scores),
        'CV R² 标准差': np.std(cv_scores)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# 可视化比较
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² 比较
x = np.arange(len(results_df))
width = 0.35

axes[0].bar(x - width/2, results_df['训练 R²'], width, label='训练 R²', alpha=0.7)
axes[0].bar(x + width/2, results_df['测试 R²'], width, label='测试 R²', alpha=0.7)
axes[0].set_xlabel('模型')
axes[0].set_ylabel('R²')
axes[0].set_title('训练 vs 测试 R²')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['模型'], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 交叉验证结果
axes[1].bar(x, results_df['CV R² 均值'], yerr=results_df['CV R² 标准差'], 
            alpha=0.7, capsize=5)
axes[1].set_xlabel('模型')
axes[1].set_ylabel('CV R²')
axes[1].set_title('交叉验证 R² (均值 ± 标准差)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df['模型'], rotation=45, ha='right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 超参数调优

### 3.1 网格搜索

In [ ]:
# Ridge 回归的超参数调优
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

ridge = Ridge()
grid_search = GridSearchCV(ridge, param_grid, cv=5, scoring='r2', return_train_score=True)
grid_search.fit(X_train_scaled, y_train)

print("Ridge 超参数调优结果")
print("=" * 40)
print(f"最佳参数: {grid_search.best_params_}")
print(f"最佳 CV R²: {grid_search.best_score_:.4f}")

In [ ]:
# 可视化调优过程
cv_results = grid_search.cv_results_

fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogx(param_grid['alpha'], cv_results['mean_train_score'], 'o-', label='训练 R²')
ax.semilogx(param_grid['alpha'], cv_results['mean_test_score'], 's-', label='验证 R²')
ax.axvline(grid_search.best_params_['alpha'], color='red', linestyle='--', 
           label=f"最佳 α = {grid_search.best_params_['alpha']}")

ax.set_xlabel('alpha (对数尺度)')
ax.set_ylabel('R²')
ax.set_title('Ridge 超参数调优')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 决策树超参数

In [ ]:
# 决策树超参数调优
param_grid_dt = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10]
}

dt = DecisionTreeRegressor(random_state=42)
grid_search_dt = GridSearchCV(dt, param_grid_dt, cv=5, scoring='r2')
grid_search_dt.fit(X_train, y_train)  # 决策树不需要标准化

print("决策树超参数调优结果")
print("=" * 40)
print(f"最佳参数: {grid_search_dt.best_params_}")
print(f"最佳 CV R²: {grid_search_dt.best_score_:.4f}")

## 4. 偏差-方差权衡

In [ ]:
# 演示偏差-方差权衡
train_sizes = np.arange(20, 181, 20)

train_errors = []
test_errors = []

for n in train_sizes:
    # 用部分数据训练
    idx = np.random.choice(len(X_train), n, replace=False)
    X_subset = X_train_scaled[idx]
    y_subset = y_train[idx]
    
    # 训练复杂模型（随机森林）
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(X_subset, y_subset)
    
    # 计算误差
    train_pred = model.predict(X_subset)
    test_pred = model.predict(X_test_scaled)
    
    train_errors.append(mean_squared_error(y_subset, train_pred))
    test_errors.append(mean_squared_error(y_test, test_pred))

print("学习曲线分析完成")

In [ ]:
# 可视化学习曲线
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(train_sizes, train_errors, 'o-', label='训练误差')
ax.plot(train_sizes, test_errors, 's-', label='测试误差')
ax.fill_between(train_sizes, train_errors, test_errors, alpha=0.2, color='red')

ax.set_xlabel('训练样本数')
ax.set_ylabel('MSE')
ax.set_title('学习曲线：偏差-方差权衡')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Research Thinking

### 问题 1：交叉验证 vs 测试集

交叉验证能代替测试集吗？

**回答：**

不能！交叉验证用于模型选择和调参，测试集用于最终评估。

正确的流程：
1. 用交叉验证选择模型和超参数
2. 在整个训练集上重新训练最佳模型
3. 在独立的测试集上评估最终性能

### 问题 2：模型复杂度选择

更复杂的模型一定更好吗？

**回答：**

不一定！需要考虑：
- **过拟合风险**：复杂模型更容易过拟合
- **数据量**：小数据用简单模型
- **可解释性**：简单模型更容易理解
- **计算成本**：复杂模型训练更慢
- **部署成本**：复杂模型需要更多资源

### 问题 3：评估指标选择

如何选择合适的评估指标？

**回答：**

- **回归**：MSE（误差大小）、R²（解释力）、MAE（鲁棒性）
- **分类**：准确率（平衡数据）、F1（不平衡数据）、AUC（排序问题）
- **业务**：根据业务目标选择（如预测需求时，误差的成本）

## 6. 练习

### 练习 1
比较不同 K 值的 K-Fold 交叉验证结果（K=3, 5, 10）。

In [ ]:
# 你的代码


### 练习 2
使用 GridSearchCV 对随机森林进行超参数调优。

In [ ]:
# 你的代码


### 练习 3
绘制不同复杂度模型的偏差-方差曲线。

In [ ]:
# 你的代码


## 7. 总结

### 本周学习要点

1. **交叉验证**：稳定评估模型性能
2. **模型比较**：选择最适合的模型
3. **超参数调优**：找到最佳配置
4. **偏差-方差权衡**：理解模型复杂度的影响

### 关键洞察

- 交叉验证比单次分割更稳定
- 没有万能的最佳模型
- 超参数调优可以显著提升性能
- 注意过拟合和泛化能力的平衡